In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "async-geotiff>=0.4",
#     "lonboard>=0.15.0",
#     "morecantile",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "pillow>=12.1.1",
#     "planetary-computer",
#     "pystac-client",
#     "sidecar>=0.8.1",
# ]
# ///

# USDA Cropland Data Layer (CDL) from Microsoft Planetary Computer in Lonboard

Visualize a [USDA Cropland Data Layer] (CDL) Cloud-Optimized GeoTIFF tile served from the [Microsoft Planetary Computer] directly in Lonboard — no tile server, fully client-side rendering via [async-geotiff] + [obstore] + [lonboard].

[USDA Cropland Data Layer]: https://www.nass.usda.gov/Research_and_Science/Cropland/SARS1a.php
[Microsoft Planetary Computer]: https://planetarycomputer.microsoft.com/dataset/usda-cdl
[async-geotiff]: https://developmentseed.org/async-geotiff/latest/
[obstore]: https://developmentseed.org/obstore/latest/
[lonboard]: https://developmentseed.org/lonboard/latest/

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and launch with:

```
uvx juv run raster-cog-cdl-pc.ipynb
```

## Imports

In [ ]:
import io
from urllib.parse import urlparse

import numpy as np
import planetary_computer
import pystac_client
from async_geotiff import GeoTIFF, Tile
from obstore.store import AzureStore
from PIL import Image
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard.raster import EncodedImage

## Find a CDL tile via the Planetary Computer STAC API

The Planetary Computer hosts CDL as many pre-tiled COGs. We use [pystac-client] with [planetary-computer]'s `sign_inplace` modifier so every returned asset href comes back with a fresh SAS token attached — no proxy server needed.

[pystac-client]: https://pystac-client.readthedocs.io/
[planetary-computer]: https://github.com/microsoft/planetary-computer-sdk-for-python

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Iowa corn belt — swap this bbox for anywhere in CONUS.
search = catalog.search(
    collections=["usda-cdl"],
    bbox=[-93.7, 41.5, -93.5, 41.7],
    query={"usda_cdl:type": {"eq": "cropland"}},
    datetime="2021",
)
item = next(search.items())
item.id, list(item.assets)

## Wire the signed Azure href into an obstore `AzureStore`

`obstore.store.AzureStore` accepts a SAS token directly. Split the signed href into `account` / `container` / `blob_path` / `sas_token` and hand obstore just the relative path.

In [ ]:
href = item.assets["cropland"].href
parsed = urlparse(href)

account = parsed.netloc.split(".")[0]                  # 'landcoverdata'
path_parts = parsed.path.lstrip("/").split("/", 1)
container, blob_path = path_parts[0], path_parts[1]    # 'usda-cdl', 'tiles/.../*.tif'
sas_token = parsed.query                                # 'sv=...&sig=...'

store = AzureStore(container, account=account, sas_token=sas_token)
geotiff = await GeoTIFF.open(blob_path, store=store)
geotiff

## Render callback

CDL is a single-band paletted raster — the COG embeds a colormap mapping each class code (corn=1, cotton=2, …) to RGB. We use the colormap to expand each fetched tile into an RGBA PNG entirely in the notebook process.

In [ ]:
cmap_array = geotiff.colormap.as_array()


def render_paletted_tile(tile: Tile) -> EncodedImage:
    arr = tile.array.data[0]  # (H, W) uint8 palette indices

    alpha = np.full(arr.shape, 255, dtype=np.uint8)
    if tile.array.nodata is not None:
        alpha[arr == tile.array.nodata] = 0
    # CDL class 0 = background
    alpha[arr == 0] = 0

    rgba = np.empty((*arr.shape, 4), dtype=np.uint8)
    rgba[..., :3] = cmap_array[arr]
    rgba[..., 3] = alpha

    img = Image.fromarray(rgba, mode="RGBA")
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

## Build the layer

In [ ]:
layer = RasterLayer.from_geotiff(geotiff, render_tile=render_paletted_tile)

In [ ]:
sidecar = Sidecar(anchor="split-right")

In [ ]:
m = Map(layer, height=800)

In [ ]:
with sidecar:
    display(m)

## Debug the render callback

Fetch one internal tile from the COG and run the render callback on it directly — useful for inspecting the RGBA output without going through the map.

In [ ]:
num_tiles_x, num_tiles_y = geotiff.tile_count
tile = await geotiff.fetch_tile(num_tiles_x // 2, num_tiles_y // 2)

In [ ]:
render_paletted_tile(tile)

## A note on the SAS token

Planetary Computer SAS tokens expire (typically ~1 hour). If the map stops loading tiles, re-run the search cell to re-sign — `obstore` doesn't auto-refresh the token. For long sessions you can wrap signing in a custom `AzureCredentialProvider`.